<a href="https://colab.research.google.com/github/avinash-tiwary/ePic/blob/main/notebooks/01_Boris_Pusher_Verification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 01: Boris Particle Pusher Verification

## 1. Theoretical Background
The non-relativistic equation of motion for a charged particle under the Lorentz force is:
64462rac{d\mathbf{x}}{dt} = \mathbf{v}, \quad mrac{d\mathbf{v}}{dt} = q\left(\mathbf{E} + \mathbf{v} 	imes \mathbf{B}ight)64462

The **Boris algorithm** (Boris, 1970) separates electric acceleration from magnetic rotation:
1. $\mathbf{v}^- = \mathbf{v}^{n-1/2} + rac{q\mathbf{E}^n}{m}rac{\Delta t}{2}$
2. $\mathbf{t} = rac{q\mathbf{B}^n}{m}rac{\Delta t}{2}, \quad \mathbf{s} = rac{2\mathbf{t}}{1 + |\mathbf{t}|^2}$
3. $\mathbf{v}' = \mathbf{v}^- + \mathbf{v}^- 	imes \mathbf{t}$
4. $\mathbf{v}^+ = \mathbf{v}^- + \mathbf{v}' 	imes \mathbf{s}$
5. $\mathbf{v}^{n+1/2} = \mathbf{v}^+ + rac{q\mathbf{E}^n}{m}rac{\Delta t}{2}$

The magnetic rotation preserves the particle kinetic energy $|\mathbf{v}|^2 = 	ext{const}$ to machine precision.

In [ ]:
# ==============================================================
# Google Colab Setup & Package Installation
# ==============================================================
import sys
if 'google.colab' in sys.modules:
    print('Running on Google Colab. Cloning repository and installing ePic...')
    !git clone https://github.com/avinash-tiwary/ePic.git
    %cd ePic
    !pip install -e .
else:
    print('Running locally. Checking ePic installation...')
    try:
        import epic
        print(f'ePic version {epic.__version__} is already available!')
    except ImportError:
        import os
        sys.path.append(os.path.abspath('..'))
        import epic
        print('Loaded ePic from relative path!')


### 2. Numerical Experiment: Pure Magnetic Gyro-Orbit
A charged particle in a uniform magnetic field $\mathbf{B} = (0, 0, B_z)$ executes circular cyclotron motion with angular frequency:
64462\omega_c = rac{q B_z}{m}, \quad T_c = rac{2\pi}{\omega_c}, \quad r_L = rac{v_\perp}{\omega_c}64462

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from epic.pusher.boris import boris_push

q = 1.0
m = 1.0
Bz = 2.0
B = np.array([0.0, 0.0, Bz])
E = np.array([0.0, 0.0, 0.0])

omega_c = q * Bz / m
T_c = 2.0 * np.pi / omega_c
r_L = 1.0 / omega_c

N_steps = 1000
dt = (2.0 * T_c) / N_steps  # Simulate exactly two gyro-periods

x = np.zeros((N_steps, 3))
v = np.zeros((N_steps, 3))
v[0] = np.array([1.0, 0.0, 0.0])

for i in range(N_steps - 1):
    v[i+1] = boris_push(v[i], E, B, q, m, dt)
    x[i+1] = x[i] + v[i+1] * dt

kinetic_energy = 0.5 * m * np.sum(v**2, axis=1)
rel_energy_drift = abs(kinetic_energy[-1] - kinetic_energy[0]) / kinetic_energy[0]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), dpi=110)
ax1.plot(x[:, 0], x[:, 1], color="#0077bb", lw=2.5, label="Boris Trajectory")
ax1.plot(0, 0, "ro", label="Guiding Center")
ax1.set_xlabel("x (Larmor Radii)")
ax1.set_ylabel("y (Larmor Radii)")
ax1.set_title(rf"Cyclotron Gyromotion ( = {T_c:.2f}\,	ext{{s}}$)")
ax1.axis("equal")
ax1.grid(True, linestyle="--", alpha=0.4)
ax1.legend()

ax2.plot(np.arange(N_steps) * dt / T_c, (kinetic_energy - kinetic_energy[0]) / kinetic_energy[0], color="#cc3311", lw=2)
ax2.set_xlabel(r"Time ( / T_c$)")
ax2.set_ylabel(r"$\Delta E_k / E_0$")
ax2.set_title(rf"Energy Conservation (Drift = {rel_energy_drift:.2e})")
ax2.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()
print(f'Boris pusher energy drift over 2 orbits: {rel_energy_drift:.2e} (Strictly machine precision!)')


### 3. ExB Drift Motion
When a transverse electric field $ is added, the guiding center drifts perpendicular to both $\mathbf{E}$ and $\mathbf{B}$ with drift velocity:
64462\mathbf{v}_E = rac{\mathbf{E} 	imes \mathbf{B}}{B^2}64462

In [ ]:
E_drift = np.array([0.0, 0.5, 0.0])
v_E_theory = np.cross(E_drift, B) / np.dot(B, B)
print(f'Theoretical ExB drift velocity: {v_E_theory}')

x_drift = np.zeros((N_steps, 3))
v_drift = np.zeros((N_steps, 3))
v_drift[0] = np.array([1.0, 0.0, 0.0])

for i in range(N_steps - 1):
    v_drift[i+1] = boris_push(v_drift[i], E_drift, B, q, m, dt)
    x_drift[i+1] = x_drift[i] + v_drift[i+1] * dt

plt.figure(figsize=(9, 5), dpi=110)
plt.plot(x_drift[:, 0], x_drift[:, 1], color="#228833", lw=2, label="Cycloid Trajectory")
plt.plot(x_drift[0, 0] + v_E_theory[0] * np.arange(N_steps) * dt,
         x_drift[0, 1] + v_E_theory[1] * np.arange(N_steps) * dt, "k--", lw=2, label="Guiding Center Drift")
plt.xlabel("x")
plt.ylabel("y")
plt.title(r"Classical $\mathbf{E} 	imes \mathbf{B}$ Drift Motion")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.show()
